In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

HEADERS = {"User-Agent": "Mozilla/5.0"}

# 🔹 India weekly chart URLs (Jan 2025)
WEEK_URLS = [
    "https://kworb.net/spotify/country/in_weekly.html?date=2025-01-03",
    "https://kworb.net/spotify/country/in_weekly.html?date=2025-01-10",
    "https://kworb.net/spotify/country/in_weekly.html?date=2025-01-17",
    "https://kworb.net/spotify/country/in_weekly.html?date=2025-01-24",
]

rows = []

for url in WEEK_URLS:
    week = url.split("date=")[-1]
    print(f"\n🔍 Scraping week: {week}")

    html = requests.get(url, headers=HEADERS).text
    soup = BeautifulSoup(html, "html.parser")

    table = soup.find("table")
    if not table:
        print("❌ No table found")
        continue

    # 🔹 Only TOP 10 songs for demo
    for tr in table.find_all("tr")[1:11]:
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        # Artist - Track
        artist_title = tds[2].text.strip()
        if " - " in artist_title:
            artist, track = artist_title.split(" - ", 1)
        else:
            artist, track = "Unknown", artist_title

        # 🔹 Find numeric streams safely
        streams = None
        for td in tds:
            txt = td.text.replace(",", "").strip()
            if re.fullmatch(r"\d+", txt):
                val = int(txt)
                if streams is None or val > streams:
                    streams = val

        if streams is None:
            continue

        rows.append([
            "India",
            week,
            track.strip(),
            artist.strip(),
            streams
        ])

df = pd.DataFrame(
    rows,
    columns=["country", "week", "track", "artist", "weekly_streams"]
)

print("\n✅ DEMO OUTPUT PREVIEW:\n")
print(df)

df.to_csv("demo_india_weekly_jan2025.csv", index=False)
print("\n📁 Saved as demo_india_weekly_jan2025.csv")



🔍 Scraping week: 2025-01-03

🔍 Scraping week: 2025-01-10

🔍 Scraping week: 2025-01-17

🔍 Scraping week: 2025-01-24

✅ DEMO OUTPUT PREVIEW:

   country        week                                              track  \
0    India  2025-01-03  Gehra Hua (w/ Arijit Singh, Irshad Kamil, Arma...   
1    India  2025-01-03                                             Sahiba   
2    India  2025-01-03  Dhurandhar - Title Track (w/ Hanumankind, Jasm...   
3    India  2025-01-03                      Haseen (w/ NDS, Rippy Grewal)   
4    India  2025-01-03                           Finding Her (w/ Bharath)   
5    India  2025-01-03                             For A Reason (w/ Ikky)   
6    India  2025-01-03  Ishq Jalakar - Karvaan (w/ Shahzad Ali, Subhad...   
7    India  2025-01-03                                              FA9LA   
8    India  2025-01-03  Apna Bana Le (w/ Arijit Singh, Amitabh Bhattac...   
9    India  2025-01-03   Shararat (w/ Madhubanti Bagchi, Jasmine Sandlas)   
10   India  

In [6]:
import pandas as pd

# 🔹 Load weekly demo data
df = pd.read_csv("demo_india_weekly_jan2025.csv")

# 🔹 Convert week to datetime
df["week"] = pd.to_datetime(df["week"])

print("🔍 Weekly data used:")
print(df)

# 🔹 MONTHLY AGGREGATION (CORE LOGIC)
monthly = (
    df
    .groupby(["track", "artist"])["weekly_streams"]
    .sum()
    .reset_index()
    .rename(columns={"weekly_streams": "jan_2025_streams"})
    .sort_values("jan_2025_streams", ascending=False)
)

# 🔹 Top 10 songs
top10_jan = monthly.head(10)

# 🔹 Save output
top10_jan.to_csv(
    "india_jan_2025_monthly_top10.csv",
    index=False
)

print("\n✅ INDIA – JAN 2025 TOP 10 SONGS:\n")
print(top10_jan)

print("\n📁 File created: india_jan_2025_monthly_top10.csv")


🔍 Weekly data used:
   country       week                                              track  \
0    India 2025-01-03  Gehra Hua (w/ Arijit Singh, Irshad Kamil, Arma...   
1    India 2025-01-03                                             Sahiba   
2    India 2025-01-03  Dhurandhar - Title Track (w/ Hanumankind, Jasm...   
3    India 2025-01-03                      Haseen (w/ NDS, Rippy Grewal)   
4    India 2025-01-03                           Finding Her (w/ Bharath)   
5    India 2025-01-03                             For A Reason (w/ Ikky)   
6    India 2025-01-03  Ishq Jalakar - Karvaan (w/ Shahzad Ali, Subhad...   
7    India 2025-01-03                                              FA9LA   
8    India 2025-01-03  Apna Bana Le (w/ Arijit Singh, Amitabh Bhattac...   
9    India 2025-01-03   Shararat (w/ Madhubanti Bagchi, Jasmine Sandlas)   
10   India 2025-01-10  Gehra Hua (w/ Arijit Singh, Irshad Kamil, Arma...   
11   India 2025-01-10                                             Sa